# Activity recognition from wearable sensors (PAMAP2): subject-wise evaluation

This notebook classifies 12 everyday and sport activities from three body-worn inertial sensors (hand, chest, ankle) and a heart-rate monitor, using the public **PAMAP2** dataset (Reiss & Stricker, 2012).

The key point is **how the model is evaluated**. Splitting data at random lets neighbouring windows from the same person appear in both the training and test sets, which gives over-optimistic results. The main result here uses **leave-one-subject-out (LOSO)** evaluation: the model is always tested on a person it has never seen.

**Run:** Runtime → Run all (about 10–20 minutes on Colab).

In [ ]:
# 1. Get the code
!git clone -q https://github.com/yashwanthajjilapuram123/Physical-activity-monitoring-project repo 2>/dev/null || true
import sys; sys.path.insert(0, 'repo')
import pamap2_pipeline as p

In [ ]:
# 2. Download PAMAP2 from the UCI Machine Learning Repository (CC BY 4.0)
import os, glob, zipfile, urllib.request
URL = 'https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip'
os.makedirs('data', exist_ok=True)
if not glob.glob('data/**/Protocol/subject101.dat', recursive=True):
    urllib.request.urlretrieve(URL, 'data/pamap2.zip')
    def unzip_all(path, dest):
        with zipfile.ZipFile(path) as z:
            z.extractall(dest)
        os.remove(path)
        for inner in glob.glob(os.path.join(dest, '**', '*.zip'), recursive=True):
            unzip_all(inner, os.path.dirname(inner))
    unzip_all('data/pamap2.zip', 'data')
PROTOCOL = os.path.dirname(glob.glob('data/**/Protocol/subject101.dat', recursive=True)[0])
print('Protocol folder:', PROTOCOL)

In [ ]:
# 3. Load, window and extract features
df = p.load_protocol(PROTOCOL)
print('Samples:', len(df), '| participants:', sorted(df.subject.unique()))
windows = p.make_windows(df)
print('Windows (5.12 s, 50% overlap):', len(windows), '| features:', windows.shape[1] - 2)
windows.groupby('subject').activity.nunique().rename('activities per participant')

In [ ]:
# 4. Evaluate: random split (optimistic) vs leave-one-subject-out (honest)
res = p.evaluate(windows)
print(p.summarise(res))
res['loso']

In [ ]:
# 5. Save results
import pandas as pd
os.makedirs('results', exist_ok=True)
res['loso'].to_csv('results/loso_results.csv', index=False)
pd.DataFrame([res['random_split']]).to_csv('results/random_split_results.csv', index=False)
print(open('results/loso_results.csv').read())

In [ ]:
# 6. Plot per-participant performance
import matplotlib.pyplot as plt
l = res['loso']
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(l.subject.astype(str), l.macro_f1, color='#0072B2', label='leave-one-subject-out')
ax.axhline(res['random_split']['macro_f1'], ls='--', color='#D55E00', label='random split (optimistic)')
ax.set_xlabel('Held-out participant'); ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1.05)
ax.legend(frameon=False); plt.tight_layout(); plt.savefig('results/loso_macro_f1.png', dpi=200)

## Notes and limitations
- Nine participants is a small sample, so per-participant results vary. Report the mean and SD across held-out participants.
- Participant 109 performed few protocol activities and has few windows.
- Features are hand-crafted (time and frequency domain). Deep learning on raw signals is a natural next step.

**Reference:** Reiss, A. & Stricker, D. (2012). Introducing a new benchmarked dataset for activity monitoring. *16th International Symposium on Wearable Computers (ISWC)*, 108–109.